## 5 — High-gap word usage summaries (overall vs MH-flagged)

## Inputs
- `data/processed/4-writestreak_posts_highgap_with_mh_embedding.csv` 
- `data/processed/1-imbault_gaps_va.csv` (or your gaps file)

## Outputs
Saved to:
- `reports/tables/`
- `reports/figures/`

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd()

# Try to use config if available; otherwise fall back to default folders.
processed_dir = REPO_ROOT / ".." / "data" / "processed"
tables_dir = REPO_ROOT / ".." / "reports" / "tables"
figures_dir = REPO_ROOT / ".." / "reports" / "figures"

try:
    import sys
    sys.path.insert(0, str(REPO_ROOT / ".." / "src"))
    from l2affect.utils.config import load_config, resolve  # type: ignore

    cfg = load_config(REPO_ROOT / ".." / "configs" / "config.yaml")
    processed_dir = resolve(cfg["paths"]["processed_dir"])
    tables_dir = resolve(cfg["paths"]["reports_tables_dir"])
    figures_dir = resolve(cfg["paths"]["reports_figures_dir"])
except Exception as e:
    print("Config not loaded (ok). Using default folders.")
    print("Reason:", e)

processed_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

processed_dir, tables_dir, figures_dir


(WindowsPath('C:/Users/hus44/Code/Directed-Reading-Project/data/processed'),
 WindowsPath('C:/Users/hus44/Code/Directed-Reading-Project/reports/tables'),
 WindowsPath('C:/Users/hus44/Code/Directed-Reading-Project/reports/figures'))

In [2]:
in_path = processed_dir / "4-writestreak_posts_highgap_with_mh_embedding.csv"
df = pd.read_csv(in_path)

print("Loaded:", in_path)
print("Rows:", len(df), "Users:", df["user_id"].nunique())
df[["post_id","mh_sim_max","mh_flag","high_gap_count","high_gap_density"]].head()

Loaded: C:\Users\hus44\Code\Directed-Reading-Project\data\processed\4-writestreak_posts_highgap_with_mh_embedding.csv
Rows: 9168 Users: 704


,post_id,mh_sim_max,mh_flag,high_gap_count,high_gap_density
0,j4p7n0,0.141932,0,1,0.015385
1,kk1866,0.354041,1,1,0.008696
2,18s52qq,0.230794,0,1,0.013333
3,18sthvq,0.255394,0,2,0.005025
4,18tq0cp,0.186911,0,1,0.125000


Define high-gap word list (has to be the same as notebook 4)

In [3]:
gaps_path = processed_dir / "1-imbault_gaps_va.csv"
gaps = pd.read_csv(gaps_path)

gaps["word"] = gaps["word"].astype(str).str.strip().str.lower()
for c in ["gap_valence","gap_arousal"]:
    gaps[c] = pd.to_numeric(gaps[c], errors="coerce")
gaps = gaps.dropna(subset=["word","gap_valence","gap_arousal"]).copy()

if "gap_mag" not in gaps.columns:
    gaps["gap_mag"] = np.sqrt(gaps["gap_valence"]**2 + gaps["gap_arousal"]**2)
else:
    gaps["gap_mag"] = pd.to_numeric(gaps["gap_mag"], errors="coerce")
    gaps = gaps.dropna(subset=["gap_mag"]).copy()

thr_top10 = float(gaps["gap_mag"].quantile(0.90))
thr_top5  = float(gaps["gap_mag"].quantile(0.95))
thr_abs15 = 1.5

THRESHOLD_MODE = "top5"   # "top10" | "top5" | "abs15"

thr_map = {"top10": thr_top10, "top5": thr_top5, "abs15": thr_abs15}
thr = thr_map[THRESHOLD_MODE]

high_set = set(gaps.loc[gaps["gap_mag"] >= thr, "word"])

print("Threshold mode:", THRESHOLD_MODE)
print("gap_mag thr:", thr)
print("High-gap vocab size:", len(high_set))

Threshold mode: top5
gap_mag thr: 2.136969107761832
High-gap vocab size: 101


Parse tokens and extract high-gap tokens per post

In [4]:
def parse_tokens(x):
    if isinstance(x, list):
        return x
    if not isinstance(x, str) or x.strip() == "":
        return []
    try:
        return json.loads(x)
    except Exception:
        return re.findall(r"[A-Za-z']+", x.lower())

df["tokens_list"] = df["tokens"].map(parse_tokens)
df["highgap_tokens"] = df["tokens_list"].map(lambda toks: [str(t).lower() for t in toks if str(t).lower() in high_set])
df["n_highgap_tokens_check"] = df["highgap_tokens"].map(len)

if "high_gap_count" in df.columns:
    mismatch = (df["n_highgap_tokens_check"] != df["high_gap_count"]).mean()
    print("Mismatch rate vs high_gap_count:", float(mismatch))

df[["high_gap_count","n_highgap_tokens_check"]].head()

Mismatch rate vs high_gap_count: 0.0


,high_gap_count,n_highgap_tokens_check
0,1,1
1,1,1
2,1,1
3,2,2
4,1,1


Word Freq Table

In [5]:
def word_stats(sub_df: pd.DataFrame, label: str) -> pd.DataFrame:
    all_tokens = []
    doc_counts = {}
    total_tokens = 0

    for toks in sub_df["highgap_tokens"]:
        toks = list(toks)
        total_tokens += len(toks)
        if len(toks) == 0:
            continue
        all_tokens.extend(toks)
        for w in set(toks):
            doc_counts[w] = doc_counts.get(w, 0) + 1

    tok_counts = pd.Series(all_tokens).value_counts() if all_tokens else pd.Series(dtype=int)
    out = tok_counts.reset_index()
    out.columns = ["word", f"{label}_token_count"]
    out[f"{label}_doc_count"] = out["word"].map(lambda w: doc_counts.get(w, 0)).astype(int)
    out[f"{label}_freq_per_10k_highgap_tokens"] = (out[f"{label}_token_count"] / max(1, total_tokens)) * 10000
    out.attrs["total_highgap_tokens"] = total_tokens
    return out

overall = word_stats(df, "overall")
mh = word_stats(df[df["mh_flag"] == 1], "mh")

print("Total high-gap tokens (overall):", overall.attrs["total_highgap_tokens"])
print("Total high-gap tokens (MH-flagged):", mh.attrs["total_highgap_tokens"])
overall.head()

Total high-gap tokens (overall): 19786
Total high-gap tokens (MH-flagged): 3583


,word,overall_token_count,overall_doc_count,overall_freq_per_10k_highgap_tokens
0,good,4907,3507,2480.036389
1,great,1606,1312,811.685030
2,story,1402,965,708.581826
3,interesting,1144,999,578.186597
4,thank,725,643,366.420702


Compare and export tables

In [ ]:
compare = overall.merge(mh, on="word", how="left")
for c in compare.columns:
    if c.startswith("mh_"):
        compare[c] = compare[c].fillna(0)

eps = 1e-9
compare["mh_enrichment_ratio"] = (compare["mh_freq_per_10k_highgap_tokens"] + eps) / (compare["overall_freq_per_10k_highgap_tokens"] + eps)

top30_mh = compare.sort_values("mh_token_count", ascending=False).head(30)

min_mh_tokens = 5
top30_enriched = compare[compare["mh_token_count"] >= min_mh_tokens].sort_values("mh_enrichment_ratio", ascending=False).head(30)

overall_out = tables_dir / f"highgap_word_freq_overall_{THRESHOLD_MODE}.csv"
mh_out = tables_dir / f"highgap_word_freq_mh_{THRESHOLD_MODE}.csv"
compare_out = tables_dir / f"highgap_word_freq_compare_{THRESHOLD_MODE}.csv"
top30_mh_out = tables_dir / f"top30_highgap_words_in_mh_{THRESHOLD_MODE}.csv"
top30_enriched_out = tables_dir / f"top30_highgap_words_mh_enriched_{THRESHOLD_MODE}.csv"

overall.to_csv(overall_out, index=False)
mh.to_csv(mh_out, index=False)
compare.to_csv(compare_out, index=False)
top30_mh.to_csv(top30_mh_out, index=False)
top30_enriched.to_csv(top30_enriched_out, index=False)

print("Saved:", overall_out)
print("Saved:", mh_out)
print("Saved:", compare_out)
print("Saved:", top30_mh_out)
print("Saved:", top30_enriched_out)

top30_mh[["word","mh_token_count","mh_doc_count","mh_freq_per_10k_highgap_tokens"]]

Saved: C:\Users\hus44\Code\Directed-Reading-Project\reports\tables\highgap_word_freq_overall_top5.csv
Saved: C:\Users\hus44\Code\Directed-Reading-Project\reports\tables\highgap_word_freq_mh_top5.csv
Saved: C:\Users\hus44\Code\Directed-Reading-Project\reports\tables\highgap_word_freq_compare_top5.csv
Saved: C:\Users\hus44\Code\Directed-Reading-Project\reports\tables\top30_highgap_words_in_mh_top5.csv
Saved: C:\Users\hus44\Code\Directed-Reading-Project\reports\tables\top30_highgap_words_mh_enriched_top5.csv


,word,mh_token_count,mh_doc_count,mh_freq_per_10k_highgap_tokens
0,good,1243.0,835.0,3469.159922
1,great,300.0,259.0,837.287190
3,interesting,177.0,160.0,493.999442
2,story,140.0,113.0,390.734022
8,normal,129.0,108.0,360.033491
4,thank,121.0,112.0,337.705833
5,due,107.0,101.0,298.632431
6,beginning,98.0,88.0,273.513815
10,special,81.0,74.0,226.067541
16,comfortable,78.0,71.0,217.694669


Plots

In [7]:
plt.figure(figsize=(10, 7))
plot_df = top30_mh.sort_values("mh_token_count", ascending=True)
plt.barh(plot_df["word"], plot_df["mh_token_count"])
plt.title(f"Top 30 high-gap words in MH-flagged posts ({THRESHOLD_MODE})")
plt.xlabel("Token count in MH-flagged posts")
plt.tight_layout()
fig1 = figures_dir / f"top30_highgap_words_in_mh_{THRESHOLD_MODE}.png"
plt.savefig(fig1, dpi=200)
print("Saved:", fig1)
plt.close()

plt.figure(figsize=(10, 7))
plot_df2 = top30_enriched.sort_values("mh_enrichment_ratio", ascending=True)
plt.barh(plot_df2["word"], plot_df2["mh_enrichment_ratio"])
plt.title(f"Top MH-enriched high-gap words (min {min_mh_tokens} MH tokens) ({THRESHOLD_MODE})")
plt.xlabel("MH enrichment ratio (MH freq / overall freq)")
plt.tight_layout()
fig2 = figures_dir / f"top30_highgap_words_mh_enriched_{THRESHOLD_MODE}.png"
plt.savefig(fig2, dpi=200)
print("Saved:", fig2)
plt.close()

Saved: C:\Users\hus44\Code\Directed-Reading-Project\reports\figures\top30_highgap_words_in_mh_top5.png
Saved: C:\Users\hus44\Code\Directed-Reading-Project\reports\figures\top30_highgap_words_mh_enriched_top5.png


Additional summaries

In [8]:
tmp = df.copy()
tmp["density_q"] = pd.qcut(tmp["high_gap_density"].fillna(0), 4, labels=["Q1","Q2","Q3","Q4"])
mh_by_q = tmp.groupby("density_q").agg(
    n_posts=("post_id","count"),
    mh_rate=("mh_flag","mean"),
    mean_mh_sim=("mh_sim_max","mean"),
).reset_index()

mh_by_q_out = tables_dir / f"mh_rate_by_highgap_density_quartile_{THRESHOLD_MODE}.csv"
mh_by_q.to_csv(mh_by_q_out, index=False)
print("Saved:", mh_by_q_out)
mh_by_q

Saved: C:\Users\hus44\Code\Directed-Reading-Project\reports\tables\mh_rate_by_highgap_density_quartile_top5.csv


,density_q,n_posts,mh_rate,mean_mh_sim
0,Q1,2311,0.221549,0.254132
1,Q2,2298,0.205396,0.254984
2,Q3,2288,0.209353,0.253631
3,Q4,2271,0.163364,0.245012


In [9]:
seed_dist = df[df["mh_flag"]==1].groupby("mh_best_seed").size().reset_index(name="n_posts").sort_values("n_posts", ascending=False)
seed_dist_out = tables_dir / f"mh_seed_distribution_{THRESHOLD_MODE}.csv"
seed_dist.to_csv(seed_dist_out, index=False)
print("Saved:", seed_dist_out)
seed_dist.head(10)

Saved: C:\Users\hus44\Code\Directed-Reading-Project\reports\tables\mh_seed_distribution_top5.csv


,mh_best_seed,n_posts
10,I’ve had trouble concentrating on things like ...,756
17,Things I usually enjoy don’t feel enjoyable la...,147
13,My eating has changed a lot recently.,128
15,My sleep has been disrupted or unrefreshing.,122
7,I’ve felt tired or like I had very little energy.,91
3,"I’ve been feeling down, depressed, or hopeless.",90
4,"I’ve felt bad about myself, like a failure, or...",86
16,Others might notice I’m moving/speaking slower...,81
0,Even small tasks feel exhausting.,60
14,My mind feels foggy and it’s hard to focus.,56


In [10]:
plt.figure(figsize=(7,5))
plt.scatter(df["high_gap_density"], df["mh_sim_max"], s=10, alpha=0.6)
plt.title(f"MH similarity vs high-gap density ({THRESHOLD_MODE})")
plt.xlabel("High-gap density")
plt.ylabel("mh_sim_max")
plt.tight_layout()
fig3 = figures_dir / f"mh_sim_vs_highgap_density_{THRESHOLD_MODE}.png"
plt.savefig(fig3, dpi=200)
print("Saved:", fig3)
plt.close()

Saved: C:\Users\hus44\Code\Directed-Reading-Project\reports\figures\mh_sim_vs_highgap_density_top5.png
